# Syria River Buffer Map (10 km)

Overview

Creates 10 km buffer zones around mapped river features in Syria.
The river data is reprojected from Web Mercator to WGS 84 / UTM zone 37N so that buffer distances can be calculated in metres.
The resulting buffer polygons are saved as a derived GeoPackage and displayed with the source rivers and administrative boundaries on an interactive map.
The buffers are not clipped to the national boundary and may extend into neighbouring countries.

シリアの河川地物を対象として、周囲10 kmのバッファを作成します。
距離をメートル単位で計算するため、河川データをWeb MercatorからWGS 84 / UTM zone 37Nへ再投影してからバッファ処理を行います。
作成したバッファポリゴンは派生GeoPackageとして保存し、元の河川および行政界とともにインタラクティブ地図上へ表示します。
バッファはシリア国境で切り抜いていないため、隣国側へ広がる場合があります。

Objectives

- Read the river and administrative boundary datasets
- Confirm the required attributes, geometries and coordinate reference systems
- Reproject the rivers to a projected coordinate system
- Create 10 km buffer polygons around the river features
- Preserve the source river attributes in the derived dataset
- Save the buffer polygons in the project output directory
- Display the rivers, buffers and administrative boundaries in Folium

- 河川データと行政界データを読み込む
- 必要な属性、ジオメトリおよび座標参照系を確認する
- 河川を投影座標系へ再投影する
- 河川地物の周囲に10 kmのバッファポリゴンを作成する
- 派生データに元の河川属性を保持する
- バッファポリゴンをプロジェクトのoutputsフォルダへ保存する
- 河川、バッファおよび行政界をFoliumで表示する

Workflow

#### English

1. Define the input and output file paths
2. Read and validate the source datasets
3. Reproject the rivers to WGS 84 / UTM zone 37N
4. Create and validate the 10 km river buffers
5. Save the derived buffers as a GeoPackage
6. Prepare lightweight copies for web-map display
7. Create the interactive map and its supporting elements
8. Save and display the HTML map

#### 日本語

1. 入出力ファイルのパスを設定する
2. 元データを読み込み、内容を確認する
3. 河川をWGS 84 / UTM zone 37Nへ再投影する
4. 10 kmの河川バッファを作成し、確認する
5. 派生バッファをGeoPackageとして保存する
6. Web地図表示用の軽量なコピーを準備する
7. インタラクティブ地図と付属要素を作成する
8. HTML地図を保存し、Notebook上に表示する

Data

River data:

- `syr_rivers_3857.geojson`
- Source: OpenStreetMap contributors

Administrative boundary data:

- `syr_admin0.geojson`
- `syr_admin1.geojson`
- Source: HDX OCHA, Syria subnational administrative boundaries

Derived data:

- `outputs/syr_river_buffer_10km.gpkg`

Technologies

- Python
- GeoPandas
- Folium
- Shapely

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

from pathlib import Path

import folium
import geopandas as gpd

In [ ]:
# 2
# Define the input and output file paths
# 入力データと出力ファイルのパスを設定する

PROJECT_DIR = Path.cwd()
ROOT_DIR = PROJECT_DIR.parents[1]

VECTOR_DIR = ROOT_DIR / "02_DATA" / "VECTOR"
OUTPUT_DIR = PROJECT_DIR / "outputs"

rivers_path = VECTOR_DIR / "syr_rivers_3857.geojson"

admin0_path = VECTOR_DIR / "syr_admin0.geojson"

admin1_path = VECTOR_DIR / "syr_admin1.geojson"

river_buffer_path = OUTPUT_DIR / "syr_river_buffer_10km.gpkg"

output_path = PROJECT_DIR / "01_syria_vector_buffer.html"

print(f"River dataset: {rivers_path}")
print(f"Derived buffer dataset: {river_buffer_path}")
print(f"Interactive map: {output_path}")

In [ ]:
# 3
# Read the source datasets
# 元となる河川データと行政界データを読み込む

rivers = gpd.read_file(rivers_path)

admin0 = gpd.read_file(admin0_path)

admin1 = gpd.read_file(admin1_path)

print(f"River features: {len(rivers):,}")
print(f"Country features: {len(admin0):,}")
print(f"Governorate features: {len(admin1):,}")

In [ ]:
# 4
# Confirm the coordinate reference systems
# 各データの座標参照系を確認する

datasets = {
    "Rivers": rivers,
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}

for dataset_name, dataset in datasets.items():

    if dataset.crs is None:
        raise ValueError(f"{dataset_name} has no defined CRS.")

    print(f"{dataset_name} CRS: {dataset.crs}")

# Confirm the CRS recorded in the source river file.
# 元の河川ファイルに設定されている座標参照系を確認する

if rivers.crs.to_epsg() != 3857:
    raise ValueError(
        "The river dataset was expected to use "
        f"EPSG:3857, but its CRS is {rivers.crs}."
    )

In [ ]:
# 5
# Check the river attributes and geometries
# 河川データの属性とジオメトリを確認する

required_river_columns = {
    "name",
    "name:en",
    "waterway",
    "osm_id",
    "geometry",
}

missing_river_columns = required_river_columns - set(rivers.columns)

if missing_river_columns:
    raise ValueError(
        "The river dataset is missing required columns: "
        f"{sorted(missing_river_columns)}"
    )

if rivers.empty:
    raise ValueError("The river dataset contains no features.")

if rivers.geometry.isna().any():
    raise ValueError("The river dataset contains missing geometries.")

if rivers.geometry.is_empty.any():
    raise ValueError("The river dataset contains empty geometries.")

if not rivers.geometry.is_valid.all():

    invalid_river_count = int((~rivers.geometry.is_valid).sum())

    raise ValueError(
        "The river dataset contains " f"{invalid_river_count:,} invalid geometries."
    )

allowed_river_geometry_types = {
    "LineString",
    "MultiLineString",
}

unexpected_river_geometry_types = (
    set(rivers.geom_type.unique()) - allowed_river_geometry_types
)

if unexpected_river_geometry_types:
    raise ValueError(
        "Unexpected river geometry types were found: "
        f"{sorted(unexpected_river_geometry_types)}"
    )

print(rivers.geom_type.value_counts())

print(
    rivers[
        [
            "name",
            "name:en",
            "waterway",
            "osm_id",
        ]
    ].head()
)

In [ ]:
# 6
# Check the administrative boundary datasets
# 行政界データの属性とジオメトリを確認する

required_admin0_columns = {
    "adm0_name",
    "adm0_pcode",
    "geometry",
}

required_admin1_columns = {
    "adm1_name",
    "adm1_pcode",
    "geometry",
}

missing_admin0_columns = required_admin0_columns - set(admin0.columns)

missing_admin1_columns = required_admin1_columns - set(admin1.columns)

if missing_admin0_columns:
    raise ValueError(
        "The country boundary dataset is missing "
        f"required columns: {sorted(missing_admin0_columns)}"
    )

if missing_admin1_columns:
    raise ValueError(
        "The governorate boundary dataset is missing "
        f"required columns: {sorted(missing_admin1_columns)}"
    )

administrative_datasets = {
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}

allowed_boundary_geometry_types = {
    "Polygon",
    "MultiPolygon",
}

for dataset_name, dataset in administrative_datasets.items():

    if dataset.empty:
        raise ValueError(f"{dataset_name} contains no features.")

    if dataset.geometry.isna().any():
        raise ValueError(f"{dataset_name} contains missing geometries.")

    if dataset.geometry.is_empty.any():
        raise ValueError(f"{dataset_name} contains empty geometries.")

    if not dataset.geometry.is_valid.all():

        invalid_geometry_count = int((~dataset.geometry.is_valid).sum())

        raise ValueError(
            f"{dataset_name} contains "
            f"{invalid_geometry_count:,} invalid geometries."
        )

    unexpected_geometry_types = (
        set(dataset.geom_type.unique()) - allowed_boundary_geometry_types
    )

    if unexpected_geometry_types:
        raise ValueError(
            f"{dataset_name} contains unexpected geometry types: "
            f"{sorted(unexpected_geometry_types)}"
        )

print(
    admin0[
        [
            "adm0_name",
            "adm0_pcode",
        ]
    ]
)

print(
    admin1[
        [
            "adm1_name",
            "adm1_pcode",
        ]
    ].sort_values("adm1_name")
)

In [ ]:
# 7
# Reproject the rivers for distance calculations
# 距離を計算するため河川を投影座標系へ再投影する

BUFFER_CRS = "EPSG:32637"
BUFFER_DISTANCE_METRES = 10_000

rivers_projected = rivers.to_crs(BUFFER_CRS)

if not rivers_projected.crs.is_projected:
    raise ValueError("The river processing CRS must be projected.")

if rivers_projected.crs.to_epsg() != 32637:
    raise ValueError("The rivers were not correctly reprojected " "to EPSG:32637.")

print(f"River processing CRS: {rivers_projected.crs}")

print("Buffer distance:", f"{BUFFER_DISTANCE_METRES / 1_000:,.0f} km")

print("Projected river bounds:", rivers_projected.total_bounds)

In [ ]:
# 8
# Create 10 km buffers around the rivers
# 河川の周囲に10 kmのバッファを作成する

river_buffers_projected = rivers_projected.copy()

river_buffers_projected["geometry"] = rivers_projected.geometry.buffer(
    BUFFER_DISTANCE_METRES
)

# Store the buffer distance as an attribute.
# バッファ距離を属性として記録する

river_buffers_projected["buffer_km"] = BUFFER_DISTANCE_METRES / 1_000

print("Created river buffers:", f"{len(river_buffers_projected):,}")

print(river_buffers_projected.geom_type.value_counts())

In [ ]:
# 9
# Check the derived river buffers
# 作成した河川バッファを確認する

if river_buffers_projected.empty:
    raise ValueError("The derived river buffer dataset contains no features.")

if len(river_buffers_projected) != len(rivers_projected):
    raise ValueError(
        "The number of buffer features does not match "
        "the number of source river features."
    )

if river_buffers_projected.crs != rivers_projected.crs:
    raise ValueError("The river buffers do not retain " "the processing CRS.")

if river_buffers_projected.geometry.isna().any():
    raise ValueError("The river buffer dataset contains " "missing geometries.")

if river_buffers_projected.geometry.is_empty.any():
    raise ValueError("The river buffer dataset contains " "empty geometries.")

if not river_buffers_projected.geometry.is_valid.all():

    invalid_buffer_count = int((~river_buffers_projected.geometry.is_valid).sum())

    raise ValueError(
        "The river buffer dataset contains "
        f"{invalid_buffer_count:,} invalid geometries."
    )

allowed_buffer_geometry_types = {
    "Polygon",
    "MultiPolygon",
}

unexpected_buffer_geometry_types = (
    set(river_buffers_projected.geom_type.unique()) - allowed_buffer_geometry_types
)

if unexpected_buffer_geometry_types:
    raise ValueError(
        "Unexpected buffer geometry types were found: "
        f"{sorted(unexpected_buffer_geometry_types)}"
    )

print("Validated buffer features:", f"{len(river_buffers_projected):,}")

print(f"Buffer CRS: {river_buffers_projected.crs}")

print("Buffer distance:", f"{river_buffers_projected['buffer_km'].iloc[0]:,.0f} km")

In [ ]:
# 10
# Save the river buffers as a GeoPackage
# 河川バッファをGeoPackageとして保存する

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

river_buffers_projected.to_file(
    river_buffer_path,
    layer="river_buffer_10km",
    driver="GPKG",
    index=False,
)

print(f"Derived river buffer saved to: {river_buffer_path}")

In [ ]:
# 11
# Read back the saved GeoPackage
# 保存したGeoPackageを読み戻して確認する

saved_river_buffers = gpd.read_file(
    river_buffer_path,
    layer="river_buffer_10km",
)

if len(saved_river_buffers) != len(river_buffers_projected):
    raise ValueError(
        "The saved buffer feature count does not " "match the derived result."
    )

if saved_river_buffers.crs != river_buffers_projected.crs:
    raise ValueError("The saved buffer CRS does not match " "the processing CRS.")

print("Saved buffer features:", f"{len(saved_river_buffers):,}")

print(f"Saved buffer CRS: {saved_river_buffers.crs}")

print(saved_river_buffers.geom_type.value_counts())

In [ ]:
# 12
# Prepare the administrative boundaries for web mapping
# 行政界をWeb地図表示用の座標参照系へ変換する

WEB_CRS = "EPSG:4326"

admin0_web = admin0.to_crs(WEB_CRS)

admin1_web = admin1.to_crs(WEB_CRS)

administrative_web_layers = {
    "Country boundaries": admin0_web,
    "Governorate boundaries": admin1_web,
}

for layer_name, layer in administrative_web_layers.items():

    if layer.crs.to_epsg() != 4326:
        raise ValueError(f"{layer_name} was not correctly " "reprojected to EPSG:4326.")

    print(f"{layer_name} CRS: {layer.crs}")

In [ ]:
# 13
# Prepare lightweight copies for web-map display
# Web地図表示用の軽量なコピーを準備する

RIVER_SIMPLIFICATION_METRES = 75
BUFFER_SIMPLIFICATION_METRES = 150

# Simplify a display copy of the rivers.
# 河川の表示用コピーを簡略化する

rivers_display_projected = rivers_projected[
    [
        "name",
        "name:en",
        "waterway",
        "osm_id",
        "geometry",
    ]
].copy()

rivers_display_projected["geometry"] = rivers_display_projected.geometry.simplify(
    RIVER_SIMPLIFICATION_METRES,
    preserve_topology=True,
)

# Merge overlapping buffers for display.
# The saved GeoPackage retains the individual features.
# 地図表示用に重複するバッファを統合する
# 保存済みGeoPackageでは個別の地物を維持する

river_buffer_display_projected = saved_river_buffers[["geometry"]].dissolve()

river_buffer_display_projected["geometry"] = (
    river_buffer_display_projected.geometry.simplify(
        BUFFER_SIMPLIFICATION_METRES,
        preserve_topology=True,
    )
)

# Repair the display geometry when simplification
# creates an invalid polygon.
# 簡略化によって不正なポリゴンが生じた場合は修復する

invalid_buffer_count_before_repair = int(
    (~river_buffer_display_projected.geometry.is_valid).sum()
)

if invalid_buffer_count_before_repair > 0:

    river_buffer_display_projected["geometry"] = (
        river_buffer_display_projected.geometry.make_valid()
    )

if not river_buffer_display_projected.geometry.is_valid.all():
    raise ValueError(
        "The buffer display layer remains invalid " "after geometry repair."
    )

allowed_display_buffer_types = {
    "Polygon",
    "MultiPolygon",
}

unexpected_display_buffer_types = (
    set(river_buffer_display_projected.geom_type.unique())
    - allowed_display_buffer_types
)

if unexpected_display_buffer_types:
    raise ValueError(
        "Unexpected buffer display geometry types "
        f"were found: {sorted(unexpected_display_buffer_types)}"
    )

# Convert the lightweight copies to WGS 84.
# 軽量化した表示用コピーをWGS 84へ変換する

rivers_display_web = rivers_display_projected.to_crs(WEB_CRS)

river_buffer_display_web = river_buffer_display_projected.to_crs(WEB_CRS)

# Repair the buffer once more if reprojection
# introduces a numerical geometry error.
# 再投影によってジオメトリエラーが生じた場合は再度修復する

if not river_buffer_display_web.geometry.is_valid.all():

    river_buffer_display_web["geometry"] = (
        river_buffer_display_web.geometry.make_valid()
    )

web_display_layers = {
    "River display layer": rivers_display_web,
    "Buffer display layer": river_buffer_display_web,
}

for layer_name, layer in web_display_layers.items():

    if layer.geometry.is_empty.any():
        raise ValueError(f"{layer_name} contains empty geometries.")

    if not layer.geometry.is_valid.all():
        raise ValueError(f"{layer_name} contains invalid geometries.")

    if layer.crs.to_epsg() != 4326:
        raise ValueError(f"{layer_name} does not use EPSG:4326.")

print(
    "Invalid buffers found after simplification:",
    f"{invalid_buffer_count_before_repair:,}",
)

print("River display features:", f"{len(rivers_display_web):,}")

print("Buffer display features:", f"{len(river_buffer_display_web):,}")

print("Individual buffer features saved:", f"{len(saved_river_buffers):,}")

print(river_buffer_display_web.geom_type.value_counts())

In [ ]:
# 14
# Create a label-free basemap focused on Syria
# 地名表記のないベースマップを作成し、シリアを表示する

m = folium.Map(
    location=[
        34.8,
        38.5,
    ],
    zoom_start=7,
    tiles=None,
)

folium.TileLayer(
    tiles=("https://{s}.basemaps.cartocdn.com/" "light_nolabels/{z}/{x}/{y}{r}.png"),
    attr=("&copy; OpenStreetMap contributors " "&copy; CARTO"),
    name="CARTO Light — No Labels",
    subdomains="abcd",
    max_zoom=20,
    overlay=False,
    control=True,
).add_to(m)

# Fit the initial view to the national boundary.
# 初期表示範囲をシリア国境へ合わせる

min_x, min_y, max_x, max_y = admin0_web.total_bounds

syria_view_bounds = [
    [min_y, min_x],
    [max_y, max_x],
]

m.fit_bounds(
    syria_view_bounds,
    padding=(35, 35),
    max_zoom=7,
)

In [ ]:
# 15
# Add the 10 km river buffer layer
# 河川から10 kmのバッファレイヤーを追加する

folium.GeoJson(
    river_buffer_display_web,
    name="River Buffer — 10 km",
    style_function=lambda feature: {
        "fillColor": "#90e0ef",
        "color": "#0077b6",
        "weight": 1,
        "fillOpacity": 0.32,
    },
    tooltip=folium.Tooltip("Area within 10 km of mapped river features"),
).add_to(m)

In [ ]:
# 16
# Add the mapped river features
# バッファの基準となる河川地物を追加する

folium.GeoJson(
    rivers_display_web,
    name="Mapped Rivers",
    style_function=lambda feature: {
        "color": "#03045e",
        "weight": 1.2,
        "opacity": 0.88,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "name:en",
            "waterway",
            "osm_id",
        ],
        aliases=[
            "River name:",
            "Waterway type:",
            "OpenStreetMap ID:",
        ],
        localize=True,
        sticky=False,
        labels=True,
    ),
).add_to(m)

In [ ]:
# 17
# Add the national and governorate boundaries
# 国境および県境レイヤーを追加する

folium.GeoJson(
    admin0_web[
        [
            "adm0_name",
            "adm0_pcode",
            "geometry",
        ]
    ],
    name="Syria Boundary",
    style_function=lambda feature: {
        "color": "#333333",
        "weight": 2.2,
        "fillOpacity": 0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm0_name",
            "adm0_pcode",
        ],
        aliases=[
            "Country:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(m)

folium.GeoJson(
    admin1_web[
        [
            "adm1_name",
            "adm1_pcode",
            "geometry",
        ]
    ],
    name="Governorate Boundaries",
    style_function=lambda feature: {
        "color": "#666666",
        "weight": 1,
        "fillOpacity": 0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
            "adm1_pcode",
        ],
        aliases=[
            "Governorate:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(m)

In [ ]:
# 18
# Add neighbouring country labels
# 周辺国名を追加する

neighbour_label_layer = folium.FeatureGroup(
    name="Neighbour Labels",
    show=True,
)

neighbour_labels = {
    "TÜRKIYE": [37.5, 37.5],
    "IRAQ": [34.5, 42.0],
    "JORDAN": [31.9, 36.5],
    "LEBANON": [34.2, 35.0],
}

for country_name, coordinates in neighbour_labels.items():

    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                width: 120px;
                margin-left: -60px;
                color: #666666;
                font-size: 14pt;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                text-shadow:
                    -1px -1px 0 white,
                    1px -1px 0 white,
                    -1px 1px 0 white,
                    1px 1px 0 white;
            ">
                {country_name}
            </div>
            """
        ),
    ).add_to(neighbour_label_layer)

neighbour_label_layer.add_to(m)

In [ ]:
# 19
# Add governorate labels
# 県名ラベルを追加する

required_label_columns = {
    "adm1_name",
    "center_lat",
    "center_lon",
}

missing_label_columns = required_label_columns - set(admin1_web.columns)

if missing_label_columns:
    raise ValueError(
        "The governorate dataset is missing label columns: "
        f"{sorted(missing_label_columns)}"
    )

label_coordinates = admin1_web[
    [
        "center_lat",
        "center_lon",
    ]
]

if label_coordinates.isna().any().any():
    raise ValueError("The governorate label coordinates " "contain missing values.")

governorate_label_layer = folium.FeatureGroup(
    name="Governorate Labels",
    show=True,
)

for _, governorate in admin1_web.iterrows():

    folium.Marker(
        location=[
            governorate["center_lat"],
            governorate["center_lon"],
        ],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                width: 120px;
                margin-left: -60px;
                color: #222222;
                font-size: 10pt;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                text-shadow:
                    -1px -1px 0 white,
                    1px -1px 0 white,
                    -1px 1px 0 white,
                    1px 1px 0 white;
            ">
                {governorate["adm1_name"]}
            </div>
            """
        ),
    ).add_to(governorate_label_layer)

governorate_label_layer.add_to(m)

In [ ]:
# 20
# Add the map information and sources
# 地図の説明、解析方法および出典を追加する

information_panel_html = f"""
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 420px;
    min-height: 225px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.25);
">
    <b style="font-size: 16px;">
        Syria
    </b>
    <br>

    <span style="
        color: #0077b6;
        font-weight: bold;
    ">
        River Buffer Map — 10 km
    </span>

    <small style="
        display: block;
        margin-top: 7px;
        line-height: 1.35;
        color: #333333;
    ">
        Buffer polygons show areas within 10 km
        of mapped river features.
        Distances were calculated in
        WGS 84 / UTM zone 37N (EPSG:32637).
        The buffers are not clipped to the Syria boundary
        and may extend into neighbouring countries.
        The map uses a dissolved and simplified display copy.
        The saved GeoPackage retains the individual
        buffer features.
    </small>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        font-size: 11px;
        line-height: 1.35;
        color: #555555;
        border-top: 1px solid #aaaaaa;
    ">
        River source:
        <a
            href="https://www.openstreetmap.org/copyright"
            target="_blank"
            style="
                color: #0077b6;
                text-decoration: none;
                font-weight: bold;
            "
        >
            OpenStreetMap contributors
        </a><br>

        Boundary source:
        <b>HDX OCHA</b><br>

        Source river features:
        <b>{len(rivers):,}</b><br>

        Derived buffer features:
        <b>{len(saved_river_buffers):,}</b><br>

        Method:
        Reprojection / 10 km Buffer / Web Simplification
    </div>
</div>
"""

m.get_root().html.add_child(folium.Element(information_panel_html))

In [ ]:
# 21
# Add the river and buffer legend
# 河川およびバッファの凡例を追加する

buffer_distance_km = BUFFER_DISTANCE_METRES / 1_000

legend_html = f"""
<div style="
    position: fixed;
    bottom: 40px;
    right: 40px;
    width: 275px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    font-size: 12px;
    z-index: 9999;
    box-shadow: 0 0 12px rgba(0, 0, 0, 0.2);
">
    <b style="font-size: 13px;">
        Map Legend
    </b>

    <div style="
        display: flex;
        align-items: center;
        margin-top: 10px;
    ">
        <span style="
            display: inline-block;
            width: 38px;
            height: 14px;
            margin-right: 9px;
            background-color: rgba(144, 224, 239, 0.45);
            border: 1px solid #0077b6;
        "></span>

        River buffer — {buffer_distance_km:,.0f} km
    </div>

    <div style="
        display: flex;
        align-items: center;
        margin-top: 9px;
    ">
        <span style="
            display: inline-block;
            width: 38px;
            height: 0;
            margin-right: 9px;
            border-top: 3px solid #03045e;
        "></span>

        Mapped river
    </div>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        color: #555555;
        border-top: 1px solid #aaaaaa;
        line-height: 1.35;
    ">
        Buffer distance:
        {BUFFER_DISTANCE_METRES:,.0f} m<br>

        Processing CRS:
        EPSG:32637<br>

        Display CRS:
        EPSG:4326
    </div>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

In [ ]:
# 22
# Add the layer control
# 地図レイヤーの表示と非表示を切り替える機能を追加する

folium.LayerControl(
    collapsed=False,
).add_to(m)

In [ ]:
# 23
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(output_path)

print(f"Map saved to: {output_path}")

m